# Day 9: Advanced Pandas Techniques

Today we'll explore advanced pandas features, custom functions, and integration with other libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Create comprehensive sample data
np.random.seed(42)
n_rows = 10000

df = pd.DataFrame({
    'id': range(n_rows),
    'customer_id': np.random.randint(1, 1000, n_rows),
    'product': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows),
    'category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home'], n_rows),
    'price': np.random.lognormal(4, 1, n_rows),
    'quantity': np.random.poisson(2, n_rows) + 1,
    'discount': np.random.beta(2, 5, n_rows),
    'date': pd.date_range('2023-01-01', periods=n_rows, freq='1H'),
    'rating': np.random.choice([1, 2, 3, 4, 5], n_rows, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
    'is_premium': np.random.choice([True, False], n_rows, p=[0.3, 0.7])
})

print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

## 1. Custom Aggregation Functions

In [ ]:
# Custom aggregation functions
def weighted_average(group):
    """Calculate weighted average of price by quantity"""
    return (group['price'] * group['quantity']).sum() / group['quantity'].sum()

def price_range(group):
    """Calculate price range"""
    return group['price'].max() - group['price'].min()

def coefficient_of_variation(group):
    """Calculate coefficient of variation"""
    return group['price'].std() / group['price'].mean()

def percentile_90(group):
    """Calculate 90th percentile"""
    return group['price'].quantile(0.9)

# Apply custom aggregations
custom_agg = df.groupby('category').agg({
    'price': [weighted_average, price_range, coefficient_of_variation, percentile_90, 'mean', 'std'],
    'quantity': ['sum', 'mean'],
    'rating': ['mean', 'median'],
    'id': 'count'
})

# Flatten column names
custom_agg.columns = ['_'.join(col).strip() for col in custom_agg.columns.values]
custom_agg = custom_agg.round(3)

print("Custom aggregations by category:")
print(custom_agg)

# Using lambda functions for quick custom aggregations
lambda_agg = df.groupby('product').agg({
    'price': [
        ('avg_price', 'mean'),
        ('price_above_100', lambda x: (x > 100).sum()),
        ('price_variance', lambda x: x.var()),
        ('price_skewness', lambda x: x.skew())
    ],
    'discount': [
        ('avg_discount', 'mean'),
        ('high_discount_count', lambda x: (x > 0.5).sum())
    ]
})

lambda_agg.columns = ['_'.join(col).strip() for col in lambda_agg.columns.values]
print("\nLambda aggregations by product:")
print(lambda_agg.round(3))

## 2. Window Functions and Rolling Operations

In [ ]:
# Sort by date for time series operations
df_ts = df.sort_values('date').reset_index(drop=True)

# Rolling window operations
df_ts['price_ma_24h'] = df_ts['price'].rolling(window=24).mean()  # 24-hour moving average
df_ts['price_ma_7d'] = df_ts['price'].rolling(window=168).mean()  # 7-day moving average
df_ts['price_std_24h'] = df_ts['price'].rolling(window=24).std()

# Expanding window operations
df_ts['price_cumulative_mean'] = df_ts['price'].expanding().mean()
df_ts['price_cumulative_max'] = df_ts['price'].expanding().max()

# Custom rolling functions
def rolling_percentile_90(series):
    return series.quantile(0.9)

df_ts['price_rolling_p90'] = df_ts['price'].rolling(window=48).apply(rolling_percentile_90)

# Rolling operations with different frequencies
df_ts_indexed = df_ts.set_index('date')
df_ts_indexed['price_daily_mean'] = df_ts_indexed['price'].resample('D').transform('mean')
df_ts_indexed['price_weekly_sum'] = df_ts_indexed['price'].resample('W').transform('sum')

print("Time series with rolling operations:")
print(df_ts[['date', 'price', 'price_ma_24h', 'price_ma_7d', 'price_std_24h']].head(50))

# Shift operations for lag/lead features
df_ts['price_lag_1'] = df_ts['price'].shift(1)
df_ts['price_lag_24'] = df_ts['price'].shift(24)
df_ts['price_lead_1'] = df_ts['price'].shift(-1)

# Calculate price changes
df_ts['price_change'] = df_ts['price'] - df_ts['price_lag_1']
df_ts['price_pct_change'] = df_ts['price'].pct_change()

print("\nPrice changes and lags:")
print(df_ts[['price', 'price_lag_1', 'price_change', 'price_pct_change']].head(10))

## 3. Advanced Indexing and MultiIndex

In [ ]:
# Create MultiIndex DataFrame
multi_df = df.groupby(['category', 'product', 'is_premium']).agg({
    'price': ['mean', 'std', 'count'],
    'quantity': 'sum',
    'rating': 'mean'
})

print("MultiIndex DataFrame:")
print(multi_df.head(10))
print()

# Accessing MultiIndex data
print("Electronics category data:")
print(multi_df.loc['Electronics'])
print()

print("Electronics, Product A data:")
print(multi_df.loc[('Electronics', 'A')])
print()

# Cross-section (xs) for advanced indexing
print("All premium products across categories:")
premium_data = multi_df.xs(True, level='is_premium')
print(premium_data.head())
print()

# Swapping index levels
swapped = multi_df.swaplevel('category', 'product')
print("After swapping index levels:")
print(swapped.head())
print()

# Unstacking MultiIndex
unstacked = multi_df.unstack('is_premium')
print("Unstacked by is_premium:")
print(unstacked.head())

# Creating custom index
df_custom_idx = df.copy()
df_custom_idx['year_month'] = df_custom_idx['date'].dt.to_period('M')
df_custom_idx['price_tier'] = pd.cut(df_custom_idx['price'], 
                                    bins=[0, 50, 100, 200, float('inf')], 
                                    labels=['Low', 'Medium', 'High', 'Premium'])

custom_multi = df_custom_idx.set_index(['year_month', 'category', 'price_tier'])
print("\nCustom MultiIndex:")
print(custom_multi.head())

## 4. Advanced Data Transformation

In [ ]:
# Transform vs Apply vs Agg
grouped = df.groupby('category')

# Transform: returns same shape as original
df['price_normalized'] = grouped['price'].transform(lambda x: (x - x.mean()) / x.std())
df['price_rank_in_category'] = grouped['price'].rank(ascending=False)
df['price_percentile'] = grouped['price'].rank(pct=True)

# Apply: can return different shapes
def category_stats(group):
    return pd.Series({
        'count': len(group),
        'avg_price': group['price'].mean(),
        'total_revenue': (group['price'] * group['quantity']).sum(),
        'top_product': group.loc[group['price'].idxmax(), 'product'],
        'avg_rating': group['rating'].mean()
    })

category_summary = df.groupby('category').apply(category_stats)
print("Category summary using apply:")
print(category_summary)
print()

# Advanced transformations with multiple columns
def calculate_metrics(group):
    group = group.copy()
    group['revenue'] = group['price'] * group['quantity']
    group['discounted_price'] = group['price'] * (1 - group['discount'])
    group['profit_margin'] = group['discount'] * 0.5  # Simplified profit calculation
    return group

df_enhanced = df.groupby('category').apply(calculate_metrics).reset_index(drop=True)

print("Enhanced data with calculated metrics:")
print(df_enhanced[['category', 'price', 'quantity', 'discount', 'revenue', 'discounted_price', 'profit_margin']].head())

# Conditional transformations
def conditional_transform(group):
    if group['is_premium'].any():
        # Premium customers get additional discount
        group['final_discount'] = group['discount'] + 0.05
    else:
        group['final_discount'] = group['discount']
    return group

df_conditional = df.groupby('customer_id').apply(conditional_transform).reset_index(drop=True)
print("\nConditional transformation results:")
print(df_conditional[['customer_id', 'is_premium', 'discount', 'final_discount']].head(10))

## 5. Working with APIs and Web Data

In [ ]:
# Simulating API data (since we can't make real API calls)
import json
from io import StringIO

# Simulate JSON API response
api_response = {
    "data": [
        {"id": 1, "name": "Product A", "price": 99.99, "category": "Electronics", 
         "specs": {"weight": 1.5, "color": "black"}},
        {"id": 2, "name": "Product B", "price": 149.99, "category": "Electronics", 
         "specs": {"weight": 2.1, "color": "white"}},
        {"id": 3, "name": "Product C", "price": 79.99, "category": "Home", 
         "specs": {"weight": 0.8, "color": "blue"}}
    ],
    "metadata": {"total": 3, "page": 1}
}

# Convert nested JSON to DataFrame
from pandas import json_normalize

# Normalize the nested JSON
api_df = json_normalize(api_response['data'])
print("API data normalized:")
print(api_df)
print()

# Handle deeply nested JSON
nested_json = '''
[
    {
        "customer": {
            "id": 1,
            "name": "John Doe",
            "address": {
                "street": "123 Main St",
                "city": "New York",
                "country": "USA"
            }
        },
        "orders": [
            {"id": 101, "amount": 250.00},
            {"id": 102, "amount": 175.50}
        ]
    }
]
'''

nested_data = json.loads(nested_json)
normalized_nested = json_normalize(nested_data, 'orders', 
                                  ['customer.id', 'customer.name', 
                                   'customer.address.city', 'customer.address.country'],
                                  errors='ignore')
print("Deeply nested JSON normalized:")
print(normalized_nested)

# Simulating web scraping results
html_table_data = '''
<table>
    <tr><th>Product</th><th>Price</th><th>Rating</th></tr>
    <tr><td>Laptop</td><td>$999</td><td>4.5</td></tr>
    <tr><td>Mouse</td><td>$25</td><td>4.2</td></tr>
    <tr><td>Keyboard</td><td>$75</td><td>4.7</td></tr>
</table>
'''

# Parse HTML table
html_df = pd.read_html(StringIO(html_table_data))[0]
print("\nHTML table parsed:")
print(html_df)

# Clean the price column
html_df['Price'] = html_df['Price'].str.replace('$', '').astype(float)
print("\nCleaned price column:")
print(html_df)

## 6. Custom Pandas Extensions

In [ ]:
# Create custom accessor
@pd.api.extensions.register_dataframe_accessor("business")
class BusinessAccessor:
    def __init__(self, pandas_obj):
        self._obj = pandas_obj
    
    def calculate_revenue(self, price_col='price', quantity_col='quantity'):
        """Calculate revenue from price and quantity"""
        return self._obj[price_col] * self._obj[quantity_col]
    
    def customer_lifetime_value(self, customer_col='customer_id', revenue_col='revenue'):
        """Calculate customer lifetime value"""
        return self._obj.groupby(customer_col)[revenue_col].sum()
    
    def abc_analysis(self, revenue_col='revenue', customer_col='customer_id'):
        """Perform ABC analysis on customers"""
        clv = self.customer_lifetime_value(customer_col, revenue_col)
        clv_sorted = clv.sort_values(ascending=False)
        
        total_revenue = clv_sorted.sum()
        cumulative_pct = clv_sorted.cumsum() / total_revenue
        
        conditions = [
            cumulative_pct <= 0.8,
            cumulative_pct <= 0.95,
            cumulative_pct <= 1.0
        ]
        choices = ['A', 'B', 'C']
        
        abc_categories = pd.Series(np.select(conditions, choices), index=clv_sorted.index)
        return abc_categories

# Use custom accessor
df['revenue'] = df.business.calculate_revenue()
clv = df.business.customer_lifetime_value()
abc_analysis = df.business.abc_analysis()

print("Using custom business accessor:")
print(f"Total revenue calculated: ${df['revenue'].sum():,.2f}")
print(f"\nTop 5 customers by CLV:")
print(clv.head())
print(f"\nABC Analysis results:")
print(abc_analysis.value_counts())

# Create custom Series accessor
@pd.api.extensions.register_series_accessor("text_analysis")
class TextAnalysisAccessor:
    def __init__(self, pandas_obj):
        self._obj = pandas_obj
    
    def word_count(self):
        """Count words in text series"""
        return self._obj.str.split().str.len()
    
    def sentiment_score(self):
        """Simple sentiment analysis (mock implementation)"""
        positive_words = ['good', 'great', 'excellent', 'amazing', 'love']
        negative_words = ['bad', 'terrible', 'awful', 'hate', 'worst']
        
        scores = []
        for text in self._obj:
            if pd.isna(text):
                scores.append(0)
                continue
            
            text_lower = str(text).lower()
            pos_count = sum(word in text_lower for word in positive_words)
            neg_count = sum(word in text_lower for word in negative_words)
            scores.append(pos_count - neg_count)
        
        return pd.Series(scores, index=self._obj.index)

# Test text analysis accessor
text_data = pd.Series([
    "This product is great and amazing",
    "Terrible quality, worst purchase ever",
    "Good value for money",
    "Average product, nothing special"
])

print("\nText analysis results:")
print("Word counts:", text_data.text_analysis.word_count().tolist())
print("Sentiment scores:", text_data.text_analysis.sentiment_score().tolist())

## 7. Integration with Machine Learning Libraries

In [ ]:
# Prepare data for machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Feature engineering
ml_df = df.copy()
ml_df['revenue'] = ml_df['price'] * ml_df['quantity']
ml_df['discounted_price'] = ml_df['price'] * (1 - ml_df['discount'])
ml_df['hour'] = ml_df['date'].dt.hour
ml_df['day_of_week'] = ml_df['date'].dt.dayofweek
ml_df['month'] = ml_df['date'].dt.month

# Encode categorical variables
le_product = LabelEncoder()
le_category = LabelEncoder()

ml_df['product_encoded'] = le_product.fit_transform(ml_df['product'])
ml_df['category_encoded'] = le_category.fit_transform(ml_df['category'])

# Select features for prediction
features = ['product_encoded', 'category_encoded', 'quantity', 'discount', 
           'hour', 'day_of_week', 'month', 'is_premium']
target = 'price'

X = ml_df[features]
y = ml_df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"MSE: {mse:.2f}")
print(f"R² Score: {r2:.3f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Add predictions back to DataFrame
test_results = X_test.copy()
test_results['actual_price'] = y_test
test_results['predicted_price'] = y_pred
test_results['prediction_error'] = abs(y_test - y_pred)

print("\nPrediction results sample:")
print(test_results[['actual_price', 'predicted_price', 'prediction_error']].head())

## 8. Advanced Data Validation and Quality Checks

In [ ]:
# Data quality assessment framework
class DataQualityChecker:
    def __init__(self, df):
        self.df = df
        self.report = {}
    
    def check_missing_values(self):
        """Check for missing values"""
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100
        
        self.report['missing_values'] = pd.DataFrame({
            'missing_count': missing,
            'missing_percentage': missing_pct
        })
        return self.report['missing_values']
    
    def check_duplicates(self):
        """Check for duplicate rows"""
        total_duplicates = self.df.duplicated().sum()
        duplicate_pct = (total_duplicates / len(self.df)) * 100
        
        self.report['duplicates'] = {
            'total_duplicates': total_duplicates,
            'duplicate_percentage': duplicate_pct
        }
        return self.report['duplicates']
    
    def check_outliers(self, columns=None):
        """Check for outliers using IQR method"""
        if columns is None:
            columns = self.df.select_dtypes(include=[np.number]).columns
        
        outlier_report = {}
        
        for col in columns:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower_bound) | (self.df[col] > upper_bound)).sum()
            outlier_pct = (outliers / len(self.df)) * 100
            
            outlier_report[col] = {
                'outlier_count': outliers,
                'outlier_percentage': outlier_pct,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound
            }
        
        self.report['outliers'] = outlier_report
        return outlier_report
    
    def check_data_types(self):
        """Check data types and suggest optimizations"""
        type_report = []
        
        for col in self.df.columns:
            dtype = self.df[col].dtype
            unique_count = self.df[col].nunique()
            unique_ratio = unique_count / len(self.df)
            
            suggestion = ""
            if dtype == 'object' and unique_ratio < 0.5:
                suggestion = "Consider converting to categorical"
            elif dtype == 'int64':
                suggestion = "Consider downcasting to int32 or int16"
            elif dtype == 'float64':
                suggestion = "Consider downcasting to float32"
            
            type_report.append({
                'column': col,
                'dtype': dtype,
                'unique_count': unique_count,
                'unique_ratio': unique_ratio,
                'suggestion': suggestion
            })
        
        self.report['data_types'] = pd.DataFrame(type_report)
        return self.report['data_types']
    
    def generate_full_report(self):
        """Generate comprehensive data quality report"""
        print("=" * 50)
        print("DATA QUALITY REPORT")
        print("=" * 50)
        
        print(f"\nDataset Shape: {self.df.shape}")
        print(f"Memory Usage: {self.df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        print("\n1. MISSING VALUES:")
        missing_report = self.check_missing_values()
        print(missing_report[missing_report['missing_count'] > 0])
        
        print("\n2. DUPLICATE ROWS:")
        dup_report = self.check_duplicates()
        print(f"Total duplicates: {dup_report['total_duplicates']} ({dup_report['duplicate_percentage']:.2f}%)")
        
        print("\n3. OUTLIERS:")
        outlier_report = self.check_outliers()
        for col, stats in outlier_report.items():
            if stats['outlier_count'] > 0:
                print(f"{col}: {stats['outlier_count']} outliers ({stats['outlier_percentage']:.2f}%)")
        
        print("\n4. DATA TYPE OPTIMIZATION:")
        type_report = self.check_data_types()
        suggestions = type_report[type_report['suggestion'] != '']
        if not suggestions.empty:
            print(suggestions[['column', 'dtype', 'suggestion']])
        else:
            print("No optimization suggestions")

# Run data quality check
quality_checker = DataQualityChecker(df)
quality_checker.generate_full_report()

## 9. Performance Monitoring and Profiling

In [ ]:
# Performance monitoring decorator
import time
import functools
import psutil
import os

def monitor_performance(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Get initial memory usage
        process = psutil.Process(os.getpid())
        initial_memory = process.memory_info().rss / 1024 / 1024  # MB
        
        # Time the function
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        
        # Get final memory usage
        final_memory = process.memory_info().rss / 1024 / 1024  # MB
        
        # Print performance metrics
        print(f"\nPerformance Report for {func.__name__}:")
        print(f"Execution time: {end_time - start_time:.4f} seconds")
        print(f"Memory usage: {initial_memory:.2f} MB -> {final_memory:.2f} MB")
        print(f"Memory change: {final_memory - initial_memory:.2f} MB")
        
        return result
    return wrapper

# Example functions to monitor
@monitor_performance
def complex_aggregation(df):
    """Complex aggregation operation"""
    return df.groupby(['category', 'product']).agg({
        'price': ['mean', 'std', 'min', 'max'],
        'quantity': ['sum', 'mean'],
        'rating': ['mean', 'count'],
        'discount': lambda x: x.quantile(0.9)
    })

@monitor_performance
def memory_intensive_operation(df):
    """Memory intensive operation"""
    # Create multiple copies and transformations
    df1 = df.copy()
    df2 = df.copy()
    df3 = df.copy()
    
    # Perform operations
    combined = pd.concat([df1, df2, df3])
    result = combined.groupby('category').agg({
        'price': 'mean',
        'quantity': 'sum'
    })
    
    return result

# Test performance monitoring
print("Testing performance monitoring:")
result1 = complex_aggregation(df)
result2 = memory_intensive_operation(df.head(1000))  # Use smaller dataset

# Benchmark different approaches
def benchmark_operations(df, n_iterations=5):
    """Benchmark different pandas operations"""
    operations = {
        'groupby_mean': lambda x: x.groupby('category')['price'].mean(),
        'pivot_table': lambda x: x.pivot_table(values='price', index='category', aggfunc='mean'),
        'query_filter': lambda x: x.query('price > 100 and rating >= 4'),
        'boolean_filter': lambda x: x[(x['price'] > 100) & (x['rating'] >= 4)],
        'sort_values': lambda x: x.sort_values(['category', 'price']),
    }
    
    results = {}
    
    for op_name, operation in operations.items():
        times = []
        
        for _ in range(n_iterations):
            start = time.time()
            _ = operation(df)
            end = time.time()
            times.append(end - start)
        
        results[op_name] = {
            'mean_time': np.mean(times),
            'std_time': np.std(times),
            'min_time': np.min(times),
            'max_time': np.max(times)
        }
    
    return pd.DataFrame(results).T

# Run benchmark
print("\nBenchmarking different operations:")
benchmark_results = benchmark_operations(df)
print(benchmark_results.round(6))

## 10. Advanced Export and Reporting

In [ ]:
# Create comprehensive business report
def generate_business_report(df, output_file='business_report.xlsx'):
    """Generate comprehensive business report in Excel"""
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Summary statistics
        summary = df.describe()
        summary.to_excel(writer, sheet_name='Summary_Statistics')
        
        # Category analysis
        category_analysis = df.groupby('category').agg({
            'price': ['count', 'mean', 'sum'],
            'quantity': 'sum',
            'rating': 'mean',
            'discount': 'mean'
        })
        category_analysis.columns = ['_'.join(col).strip() for col in category_analysis.columns.values]
        category_analysis.to_excel(writer, sheet_name='Category_Analysis')
        
        # Product performance
        product_performance = df.groupby('product').agg({
            'price': 'mean',
            'quantity': 'sum',
            'rating': 'mean'
        }).round(2)
        product_performance.to_excel(writer, sheet_name='Product_Performance')
        
        # Time series analysis
        daily_sales = df.groupby(df['date'].dt.date).agg({
            'price': 'sum',
            'quantity': 'sum'
        })
        daily_sales['revenue'] = daily_sales['price'] * daily_sales['quantity']
        daily_sales.to_excel(writer, sheet_name='Daily_Sales')
        
        # Customer analysis
        customer_analysis = df.groupby('customer_id').agg({
            'price': ['count', 'sum', 'mean'],
            'quantity': 'sum',
            'rating': 'mean'
        })
        customer_analysis.columns = ['_'.join(col).strip() for col in customer_analysis.columns.values]
        customer_analysis = customer_analysis.sort_values('price_sum', ascending=False).head(100)
        customer_analysis.to_excel(writer, sheet_name='Top_Customers')
    
    print(f"Business report saved to {output_file}")

# Generate the report
generate_business_report(df)

# Create formatted HTML report
def create_html_report(df):
    """Create formatted HTML report"""
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Business Analytics Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; }}
            h1 {{ color: #333; }}
            h2 {{ color: #666; }}
            table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #f2f2f2; }}
            .metric {{ background-color: #e7f3ff; padding: 10px; margin: 10px 0; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>Business Analytics Report</h1>
        <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <h2>Key Metrics</h2>
        <div class="metric">Total Records: {len(df):,}</div>
        <div class="metric">Total Revenue: ${(df['price'] * df['quantity']).sum():,.2f}</div>
        <div class="metric">Average Order Value: ${df['price'].mean():.2f}</div>
        <div class="metric">Average Rating: {df['rating'].mean():.2f}/5</div>
        
        <h2>Category Performance</h2>
        {df.groupby('category')['price'].agg(['count', 'mean', 'sum']).round(2).to_html()}
        
        <h2>Product Analysis</h2>
        {df.groupby('product').agg({{'price': 'mean', 'rating': 'mean', 'quantity': 'sum'}}).round(2).to_html()}
        
    </body>
    </html>
    """
    
    with open('business_report.html', 'w') as f:
        f.write(html_content)
    
    print("HTML report saved to business_report.html")

# Generate HTML report
create_html_report(df)

# Export data in multiple formats
export_formats = {
    'csv': df.to_csv,
    'json': lambda filename: df.to_json(filename, orient='records', date_format='iso'),
    'parquet': df.to_parquet if hasattr(df, 'to_parquet') else None
}

print("\nExporting data in multiple formats:")
for format_name, export_func in export_formats.items():
    if export_func:
        try:
            filename = f'data_export.{format_name}'
            if format_name == 'csv':
                export_func(filename, index=False)
            elif format_name == 'parquet':
                export_func(filename, index=False)
            else:
                export_func(filename)
            print(f"✓ Exported to {filename}")
        except Exception as e:
            print(f"✗ Failed to export to {format_name}: {e}")

print("\nAdvanced pandas tutorial series completed!")
print("You now have comprehensive knowledge of pandas for data analysis.")

## Practice Exercises

In [ ]:
# Exercise 1: Build a complete data pipeline
def data_pipeline(raw_data):
    """Complete data processing pipeline"""
    # 1. Data cleaning
    cleaned_data = raw_data.copy()
    cleaned_data = cleaned_data.dropna()
    cleaned_data = cleaned_data.drop_duplicates()
    
    # 2. Feature engineering
    cleaned_data['revenue'] = cleaned_data['price'] * cleaned_data['quantity']
    cleaned_data['discounted_price'] = cleaned_data['price'] * (1 - cleaned_data['discount'])
    cleaned_data['profit_margin'] = cleaned_data['discount'] * 0.3  # Simplified
    
    # 3. Aggregations
    summary = cleaned_data.groupby('category').agg({
        'revenue': ['sum', 'mean'],
        'rating': 'mean',
        'quantity': 'sum'
    })
    
    # 4. Advanced analytics
    customer_segments = cleaned_data.business.abc_analysis()
    
    return {
        'cleaned_data': cleaned_data,
        'summary': summary,
        'customer_segments': customer_segments
    }

# Run the pipeline
pipeline_results = data_pipeline(df)
print("Data pipeline completed successfully!")
print(f"Processed {len(pipeline_results['cleaned_data'])} records")
print(f"Customer segments: {pipeline_results['customer_segments'].value_counts().to_dict()}")

# Exercise 2: Create a real-time data processor
class RealTimeProcessor:
    def __init__(self, window_size=100):
        self.window_size = window_size
        self.buffer = pd.DataFrame()
        self.metrics = {}
    
    def process_batch(self, new_data):
        """Process new batch of data"""
        # Add to buffer
        self.buffer = pd.concat([self.buffer, new_data]).tail(self.window_size)
        
        # Calculate rolling metrics
        self.metrics = {
            'avg_price': self.buffer['price'].mean(),
            'total_quantity': self.buffer['quantity'].sum(),
            'avg_rating': self.buffer['rating'].mean(),
            'revenue': (self.buffer['price'] * self.buffer['quantity']).sum()
        }
        
        return self.metrics
    
    def get_alerts(self):
        """Generate alerts based on metrics"""
        alerts = []
        
        if self.metrics.get('avg_rating', 5) < 3.0:
            alerts.append("Low average rating detected!")
        
        if self.metrics.get('avg_price', 0) > 200:
            alerts.append("High average price detected!")
        
        return alerts

# Test real-time processor
processor = RealTimeProcessor(window_size=50)

# Simulate streaming data
for i in range(0, 200, 20):
    batch = df.iloc[i:i+20]
    metrics = processor.process_batch(batch)
    alerts = processor.get_alerts()
    
    if i % 60 == 0:  # Print every 3rd batch
        print(f"\nBatch {i//20 + 1} metrics:")
        for key, value in metrics.items():
            print(f"  {key}: {value:.2f}")
        
        if alerts:
            print("  Alerts:", alerts)

print("\nReal-time processing simulation completed!")

## Tomorrow's Preview
In Day 10, we'll cover:
- Complete Project Walkthrough
- Best Practices and Design Patterns
- Production-Ready Code
- Final Capstone Project